# 07 · External test: the 10 new boreholes, scored once

| Item | Setting |
|---|---|
| Boreholes | `External_BHs_SR_Elev.csv`; the 3 boreholes that are also in the main file are removed (they are training data); 10 new Downtown boreholes remain |
| Model | the **frozen** final model of notebook 05 (fingerprint checked) and the depth-only reference. Nothing is tuned or refitted |
| Cells | same rules as the training data, including the SS rule (unclassified SS samples between two samples of the same class are filled) |
| Scores | all cells (main) and measured cells only; all boreholes and by distance to the nearest training borehole |
| Once | the result is saved with the model's fingerprint. Running again shows the saved result; it does not score again |

**Disclose in the thesis:** these boreholes were already scored once with the V3 model, and the decision to add the 5 km ring was made after that. V4 never used them for training, tuning or any choice.

In a **quick test run** the new boreholes are never scored: 8 training boreholes stand in for them.

**Run time:** under 1 minute.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])

model, fin, rec = H.load_final(S)
ref, ref_rec, _ = H.load_final(S, H.DEPTH_ONLY)
print("Frozen model:", fin["model"], "| threshold", fin["threshold"], "| SHA-256", fin["sha256"][:16])
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
train_xy = cells.groupby("BH_Code")[["UTM_Easting_m", "UTM_Northing_m"]].first().values
OUT, PRIV = H.out_dir(S, "07_new_external"), H.out_dir(S, "07_new_external", private=True)
record = OUT / "new_external_result.json"

## 1. The new boreholes (counts only, no labels yet)

In [ ]:
new_records, info = H.load_new_external(S)
print(info)
new, info_c = H.make_cells(new_records, S)
new, info_f = H.fill_gaps(new, new_records, S)
bh = new.groupby("BH_Code").agg(cells=("y", "size"), measured=("measured", "sum"),
                                E=("UTM_Easting_m", "first"), N=("UTM_Northing_m", "first"))
bh["dist_nearest_train_m"] = H.nearest_distance(bh[["E", "N"]].values, train_xy).round(0)
display(bh[["cells", "measured", "dist_nearest_train_m"]])
print(f"{len(new)} cells ({int(new.measured.sum())} measured + {int((new.measured == 0).sum())} filled)")

## 2. Score (once)

In [ ]:
BANDS, LABELS = [0, 300, 700, 1000, np.inf], ["< 300 m", "300-700 m", "700-1000 m", "> 1000 m"]
if S["run"]["quick"]:
    stand_in = cells.BH_Code.drop_duplicates().sample(8, random_state=1)
    test = cells[cells.BH_Code.isin(stand_in)].copy()
    label = "QUICK STAND-IN (training boreholes; the new boreholes are not scored)"
elif record.exists():
    saved = json.loads(record.read_text())
    if saved["model_sha256"] != fin["sha256"]:
        raise RuntimeError("The new boreholes were scored with another model; they are not scored again.")
    print("Already scored on", saved["date"], "- showing the saved result.")
    test = None
else:
    test = new.copy()
    label = "NEW EXTERNAL BOREHOLES (V4 frozen model)"

if test is not None:
    test["dist_nearest_train_m"] = H.nearest_distance(test[["UTM_Easting_m", "UTM_Northing_m"]].values, train_xy)
    test["group"] = pd.cut(test.dist_nearest_train_m, BANDS, right=False, labels=LABELS).astype(str)
    out = []
    for name, m, thr in ((fin["model"], model, fin["threshold"]), (H.DEPTH_ONLY, ref, ref_rec["threshold"])):
        p = m.p_clay(test)
        t = test.assign(model=name, p_clay=p, y_pred=(p >= thr).astype(int))
        out.append(t)
    scored = pd.concat(out, ignore_index=True)
    rows = []
    for (name, ), g in scored.groupby(["model"]):
        for subset, d in (("all", g), ("measured", g[g.measured == 1])):
            rows.append({"model": name, "cells": subset, "group": "all", **H.scores(d.y, d.p_clay, d.y_pred)})
            for grp, dd in d.groupby("group"):
                rows.append({"model": name, "cells": subset, "group": grp, **H.scores(dd.y, dd.p_clay, dd.y_pred)})
    res = pd.DataFrame(rows)
    scored.to_csv(PRIV / "new_external_predictions.csv", index=False)
    if not S["run"]["quick"]:
        record.write_text(json.dumps({"label": label, "date": time.strftime("%Y-%m-%d %H:%M"),
                                      "model": fin["model"], "model_sha256": fin["sha256"], "data": info,
                                      "cells": {"all": len(new), "measured": int(new.measured.sum())},
                                      "scores": res.to_dict("records")}, indent=1, default=str))
    print(label)
else:
    res = pd.DataFrame(json.loads(record.read_text())["scores"])
    scored = pd.read_csv(PRIV / "new_external_predictions.csv")
cols = ["model", "cells", "group", "n", "accuracy", "balanced_accuracy", "macro_f1", "recall_clay", "recall_sand",
        "roc_auc", "brier"]
display(res[res.group == "all"][cols].round(3))
display(res[res.group != "all"][cols].round(3))

## 3. By borehole and figure

In [ ]:
per = scored.groupby(["model", "BH_Code"]).apply(lambda g: pd.Series({
    "n": len(g), "clay_share": g.y.mean(), "accuracy": (g.y == g.y_pred).mean(),
    "dist_nearest_train_m": round(g.dist_nearest_train_m.iloc[0])}), include_groups=False).reset_index()
per = per.pivot_table(index=["BH_Code", "dist_nearest_train_m", "n", "clay_share"], columns="model",
                      values="accuracy").reset_index().sort_values("dist_nearest_train_m")
display(per.round(3)); per.to_csv(PRIV / "new_external_by_borehole.csv", index=False)
a = res[(res.group == "all") & (res.cells == "all")].set_index("model")
fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(4)
for j, (name, col) in enumerate(((fin["model"], "#A0004E"), (H.DEPTH_ONLY, "#9E9E9E"))):
    v = a.loc[name, ["macro_f1", "balanced_accuracy", "roc_auc", "brier"]].values.astype(float)
    ax.bar(x + (j - 0.5) * 0.38, v, width=0.36, color=col, label=name)
    for xi, vi in zip(x + (j - 0.5) * 0.38, v):
        ax.text(xi, vi + 0.01, f"{vi:.2f}", ha="center", fontsize=7)
ax.set_xticks(x, ["Macro-F1", "Balanced acc.", "ROC-AUC", "Brier (lower = better)"], fontsize=8)
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=8); ax.set_title("New external boreholes, all cells", fontsize=10)
fig.savefig(OUT / "fig_new_external.png", dpi=200, bbox_inches="tight")
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 08.")